In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.stats import linregress


# Define file paths: 
> **data_filelaCO** = data from plate reader: Copy and paste into an excel file the raw table of absorbance reads given by plate reader. Note the rows where the table starts. Ideally start at A1.

> **layout_file**  = is a csv file that contains the well:sample definition. This can be very different to the SEC plate due to standards and can be exported directly from Benchling if using the protocol. 
  Otherwise, produce a table with paired well name and sample name and comment the well_to_sample_map.
  
> **Concentration_data** - Concentration of each standard loaded onto the plate. Use same label as plate layout - more specification and an example is given on the benchling protocol  https://benchling.com/s/prt-Ia4VDTY8MrHeVpU7a8xT?m=slm-VZTe4mU62ZmlRfQ68yIk

In [ ]:
# --- Portable paths (replaces machine-specific os.chdir) ---------------
# Resolve the deposit root from wherever the notebook is launched so it
# runs anywhere. Intermediate/derived files are written to code/outputs/.
from pathlib import Path
_cwd = Path.cwd()
REPO = next((p for p in [_cwd, *_cwd.parents]
             if (p / 'construct').is_dir() and (p / 'examples').is_dir()), _cwd)
OUTDIR = REPO / 'code' / 'outputs'
OUTDIR.mkdir(parents=True, exist_ok=True)
print('Deposit root:', REPO)
print('Outputs ->', OUTDIR)
ELL = REPO / 'data' / 'Figure5_source_data' / 'ellmans_raw'

In [ ]:
# File paths
data_path = str(ELL / '20240607_ellmans_secpep_gg.xlsx')
layout_file = str(ELL / '20240610_layout.csv')
concentration_data = pd.read_csv(ELL / 'standards.csv')


# Load the data
data = pd.read_excel(data_path, usecols="A:CS", skiprows=2, nrows=302, header=0) #Check the table distribution 
data.set_index('Wavelength', inplace=True)

# Now 'data' DataFrame has Wavelength as the index and row 24 from the Excel file as headers
print("Data loaded with Wavelength as index and proper headers:")
print(data.head())

### Define well distribution and average data from replicates
    *(if not in a 1:1 column format)*

In [ ]:
# Process sample layout from CSV to create well_to_sample_map
def process_sample_layout(layout_file):
    sample_layout = pd.read_csv(layout_file, header=0, index_col=0)
    rows = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    cols = [str(i) for i in range(1, 13)]
    well_data = []
    for row in rows:
        for col in cols:
            sample = sample_layout.at[row, col] if col in sample_layout.columns and row in sample_layout.index else 'NaN'
            well_data.append({'Well': f'{row}{col}', 'Sample': sample})
    return pd.DataFrame(well_data)

# Process the layout file to create well_to_sample_map if needed (assuming this part is clear)
well_df = process_sample_layout(layout_file)
well_to_sample_map = dict(zip(well_df['Well'], well_df['Sample']))

# Relabel the columns based on the well_to_sample_map
data.columns = [well_to_sample_map.get(col, col) for col in data.columns]

# Remove columns where the sample name is NaN
data = data.loc[:, ~data.columns.isna()]  # This ensures only non-NaN columns are retained

# Additionally, you might have string 'NaN' which should be handled
data = data.loc[:, data.columns != 'NaN']

# Transpose data for easier manipulation
data = data.T

# Group by sample names (which are now the index after transpose) and calculate the mean
# This effectively averages the absorbance readings for each sample across its replicates
averaged_data = data.groupby(data.index).mean()

# Transpose back if needed for easier plotting or analysis
averaged_data = averaged_data.T

# Print to check the processed data
print("Processed Averaged Data:")
print(averaged_data.head())

### Visualise the absorbance scpectra of the sample

In [ ]:
def plot_absorbance(averaged_data):
    # Create the figure and axis for plotting
    plt.figure(figsize=(12, 8))
    
    # Generate a color map to give each sample a unique color
    cmap = plt.get_cmap('viridis')  # You can choose any colormap you like
    colors = cmap(np.linspace(0, 1, len(averaged_data.columns)))

    # Plot each sample's averaged absorbance across the wavelengths
    for i, column in enumerate(averaged_data.columns):
        plt.plot(averaged_data.index, averaged_data[column], label=column, color=colors[i])
    
    # Add legend, labels, and title to the plot
    plt.legend(title='Samples', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Absorbance')
    plt.title('Averaged Absorbance Readings Across Different Wavelengths')
    plt.grid(True)
    plt.tight_layout()  # Adjust layout to not overlap elements
    plt.show()

# Assuming 'averaged_data' is your DataFrame after averaging across replicates
plot_absorbance(averaged_data)

### Filter data from the wavelenghts needed (408 nm)

In [ ]:
# Filter the data to include only wavelengths between 405 nm and 420 nm
filtered_data = averaged_data.loc[(averaged_data.index >= 406) & (averaged_data.index <= 412)]

def plot_filtered_absorbance(filtered_data):
    # Create the figure and axis for plotting
    plt.figure(figsize=(12, 8))
    
    # Generate a color map to give each sample a unique color
    cmap = plt.get_cmap('viridis')  # You can choose any colormap you like
    colors = cmap(np.linspace(0, 1, len(filtered_data.columns)))

    # Plot each sample's averaged absorbance across the wavelengths
    for i, column in enumerate(filtered_data.columns):
        plt.plot(filtered_data.index, filtered_data[column], label=column, color=colors[i])
    
    # Add legend, labels, and title to the plot
    plt.legend(title='Samples', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Absorbance')
    plt.title('Averaged Absorbance Readings Between 405 nm and 420 nm')
    plt.grid(True)
    plt.tight_layout()  # Adjust layout to not overlap elements
    plt.show()

# Call the function to plot the filtered data
plot_filtered_absorbance(filtered_data)


### Find the maximum absorbance reads per sample

In [ ]:
def find_max_absorbance(filtered_data):
    # Find the maximum absorbance value in each column and the corresponding wavelength
    max_values = {}
    for column in filtered_data.columns:
        max_index = filtered_data[column].idxmax()  # Get the index (wavelength) of the max absorbance
        max_absorbance = filtered_data[column].max()  # Get the max absorbance value
        max_values[column] = (max_index, max_absorbance)  # Store in a dictionary

    return max_values

# Calculate max absorbance values for the filtered data
max_absorbance_values = find_max_absorbance(filtered_data)

# Print the results
print("Maximum Absorbance Values and Their Wavelengths:")
for sample, values in max_absorbance_values.items():
    print(f"{sample}: Maximum absorbance of {values[1]:.3f} at {values[0]} nm")


### Calculate the standard curve
> **concentration_data** = a csv file with a table defining name and standard concentrations

> This cell plots all concentration to visualise the curve behaivor

In [ ]:
# Assuming 'averaged_data' is your DataFrame and you have wavelengths as the index
absorbance_at_410 = averaged_data.loc[410]

# Map the absorbance to the corresponding concentrations
# 'Standard' in concentration_data matches columns in absorbance_at_410
concentration_data['Absorbance'] = concentration_data['Standard'].map(absorbance_at_410)

# Drop any rows where Absorbance is NaN (this might happen if some standards are not in your absorbance data)
concentration_data.dropna(subset=['Absorbance'], inplace=True)

# Perform linear regression
slope, intercept, r_value, p_value, std_err = linregress(concentration_data['uM_Concentration'], concentration_data['Absorbance'])

# Print the regression results
print(f"Slope: {slope}")
print(f"Intercept: {intercept}")
print(f"R-squared: {r_value**2}")

# Plot the standard curve
plt.figure(figsize=(10, 6))
plt.plot(concentration_data['uM_Concentration'], concentration_data['Absorbance'], 'o', label='Data points')
plt.plot(concentration_data['uM_Concentration'], intercept + slope * concentration_data['uM_Concentration'], 'r', label='Fit: y={:.2f}x+{:.2f}'.format(slope, intercept))
plt.xlabel('Concentration (uM)')
plt.ylabel('Absorbance at 408 nm')
plt.title('Standard Curve')
plt.legend()
plt.grid(True)
plt.show()


### Filter standards used (Useful for unknonw concentrations and wide standard curves)

In [ ]:

#DEFINING STANDARDS TO USE TO GET CONCENTRATION: 

# Filter out higher concentration standards (e.g., greater than S04)
concentration_data_filtered = concentration_data[concentration_data['Standard'].isin(['S01','S02','S03','S04','S05', 'S06', 'S07', 'S08','NC1'])]

# Drop any rows where Absorbance is NaN (this might happen if some standards are not in your absorbance data)
concentration_data_filtered.dropna(subset=['Absorbance'], inplace=True)

# Perform linear regression
slope, intercept, r_value, p_value, std_err = linregress(concentration_data_filtered['uM_Concentration'], concentration_data_filtered['Absorbance'])

# Print the regression results
print(f"Slope: {slope}")
print(f"Intercept: {intercept}")
print(f"R-squared: {r_value**2}")

# Plot the standard curve
plt.figure(figsize=(10, 6))
plt.plot(concentration_data_filtered['uM_Concentration'], concentration_data_filtered['Absorbance'], 'o', label='Data points')
plt.plot(concentration_data_filtered['uM_Concentration'], intercept + slope * concentration_data_filtered['uM_Concentration'], 'r', label='Fit: y={:.2f}x+{:.2f}'.format(slope, intercept))
plt.xlabel('Concentration (uM)')
plt.ylabel('Absorbance at 408 nm')
plt.title('Standard Curve')
plt.legend()
plt.grid(True)
plt.show()


## DETERMINING THE UNKNOWN CONCENTRATION (with defined standards)

In [ ]:
concentration_data['Absorbance'] = concentration_data['Standard'].map(absorbance_at_410)

### *Define the subset of standards you are interested in* #####
subset_standards = ['S01','S02','S03','S04','S05', 'S06', 'S07', 'S08','NC']
standard_data = concentration_data[concentration_data['Standard'].isin(subset_standards)]

# Drop any rows where Absorbance is NaN
standard_data.dropna(subset=['Absorbance'], inplace=True)

# Perform linear regression on the standard data
slope, intercept, r_value, p_value, std_err = linregress(standard_data['uM_Concentration'], standard_data['Absorbance'])

# Identify all absorbance data entries that are not in the known standard list
unknowns_absorbance = absorbance_at_410[~absorbance_at_410.index.isin(concentration_data['Standard'])]

# Calculate concentrations for the unknowns with the regression model
unknowns_concentration = (unknowns_absorbance - intercept) / slope

# Display unknowns and their calculated concentrations
print("Unknowns with Calculated Concentrations:")
print(unknowns_concentration)


# Print the regression results
print(f"Slope: {slope}")
print(f"Intercept: {intercept}")
print(f"R-squared: {r_value**2}")

# Save the data to a CSV file
standard_data.to_csv(OUTDIR / 'standard_output.csv', index=False)
unknowns_concentration.to_csv(OUTDIR / 'unknowns_output.csv')



# Plotting linear regression
plt.figure(figsize=(10, 6))
plt.plot(standard_data['uM_Concentration'], standard_data['Absorbance'], 'o', label='Standard Data Points')
plt.plot(unknowns_concentration, unknowns_absorbance, 'x', label='Unknowns', color='purple')
plt.plot([0, max(standard_data['uM_Concentration'])], [intercept + slope * 0, intercept + slope * max(standard_data['uM_Concentration'])], 'r-', label='Fit Line')
plt.xlabel('Concentration (uM)')
plt.ylabel('Absorbance at 408 nm')
plt.title('Standard Curve with Unknowns')
plt.legend()
plt.grid(True)
plt.savefig(OUTDIR / 'standard_curve_with_unknowns.png')
plt.show()

